AVIRAL SAXENA | Roll No: 2025AG05387 | DM4ML Assignment 2

In [ ]:
import os
import sys
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/recommendation-pipeline"
SRC_DIR = os.path.join(PROJECT_DIR, "src")

sys.path.insert(0, PROJECT_DIR)

print("Project directory:", PROJECT_DIR)
print("Source directory exists:", os.path.exists(SRC_DIR))

Mounted at /content/drive
Project directory: /content/drive/MyDrive/recommendation-pipeline
Source directory exists: True


In [ ]:
import os

ingestion_code = '''
import os
import pandas as pd

def load_raw_data(project_dir):
    """Load MovieLens datasets from CSV files."""

    raw_dir = os.path.join(project_dir, "data", "raw")

    datasets = {}

    for name in ["movies", "ratings", "tags", "links", "users"]:
        file_path = os.path.join(raw_dir, f"{name}.csv")
        datasets[name] = pd.read_csv(file_path)

    return datasets


def save_processed_data(datasets, project_dir):
    """Save datasets in Parquet format."""

    processed_dir = os.path.join(project_dir, "data", "processed")
    os.makedirs(processed_dir, exist_ok=True)

    for name, dataframe in datasets.items():
        output_path = os.path.join(
            processed_dir, f"{name}.parquet"
        )
        dataframe.to_parquet(output_path, index=False)

    return True
'''

module_path = os.path.join(
    SRC_DIR, "ingestion", "data_loader.py"
)

with open(module_path, "w") as file:
    file.write(ingestion_code)

print("Data ingestion module created successfully!")

Data ingestion module created successfully!


In [ ]:
#ingestion module
from src.ingestion.data_loader import load_raw_data

datasets = load_raw_data(PROJECT_DIR)

print("Data ingestion module test results:\n")

for name, dataframe in datasets.items():
    print(f"{name}: {dataframe.shape}")

Data ingestion module test results:

movies: (9742, 3)
ratings: (100836, 4)
tags: (3683, 4)
links: (9742, 3)
users: (610, 4)


In [ ]:
# Data cleaning module
import os

cleaning_code = '''
import pandas as pd

def clean_dataset(dataframe):
    """Remove duplicate records and rows with missing values."""
    cleaned = dataframe.copy()
    cleaned = cleaned.drop_duplicates()
    cleaned = cleaned.dropna()
    return cleaned.reset_index(drop=True)
'''

module_path = os.path.join(
    SRC_DIR, "preparation", "clean_data.py"
)

with open(module_path, "w") as file:
    file.write(cleaning_code)

print("Data cleaning module created successfully!")

Data cleaning module created successfully!


In [ ]:
#Time-Based Split module
split_code = '''
import pandas as pd

def time_based_split(ratings_df, test_size=0.2):
    """Split ratings chronologically into training and testing datasets."""

    sorted_df = ratings_df.sort_values(
        "timestamp", kind="stable"
    ).reset_index(drop=True)

    split_index = int(len(sorted_df) * (1 - test_size))

    train_df = sorted_df.iloc[:split_index].copy()
    test_df = sorted_df.iloc[split_index:].copy()

    return train_df, test_df
'''

module_path = os.path.join(
    SRC_DIR, "preparation", "split_data.py"
)

with open(module_path, "w") as file:
    file.write(split_code)

print("Time-based split module created successfully!")

Time-based split module created successfully!


In [ ]:
#Data Transformation module
transformation_code = '''
import pandas as pd

def transform_ratings(ratings_df, movies_df, users_df):
    """Add date features and movie/user information."""

    transformed = ratings_df.copy()

    transformed["datetime"] = pd.to_datetime(
        transformed["timestamp"], unit="s"
    )

    transformed["year"] = transformed["datetime"].dt.year
    transformed["month"] = transformed["datetime"].dt.month
    transformed["day"] = transformed["datetime"].dt.day

    transformed = transformed.merge(
        movies_df, on="movieId", how="left"
    )

    transformed = transformed.merge(
        users_df, on="userId", how="left"
    )

    return transformed
'''

module_path = os.path.join(
    SRC_DIR, "transformation", "transform_data.py"
)

with open(module_path, "w") as file:
    file.write(transformation_code)

print("Data transformation module created successfully!")

Data transformation module created successfully!


In [ ]:
from src.preparation.clean_data import clean_dataset
from src.preparation.split_data import time_based_split
from src.transformation.transform_data import transform_ratings

ratings_clean = clean_dataset(datasets["ratings"])
movies_clean = clean_dataset(datasets["movies"])
users_clean = clean_dataset(datasets["users"])

train_df, test_df = time_based_split(
    ratings_clean, test_size=0.2
)

train_transformed = transform_ratings(
    train_df, movies_clean, users_clean
)

test_transformed = transform_ratings(
    test_df, movies_clean, users_clean
)

print("Cleaned ratings:", ratings_clean.shape)
print("Training records:", train_df.shape)
print("Testing records:", test_df.shape)

print("\nTransformed datasets:")
print("Training:", train_transformed.shape)
print("Testing:", test_transformed.shape)

print("\nChronological split verification:")
print(
    "Training maximum timestamp:",
    train_df["timestamp"].max()
)
print(
    "Testing minimum timestamp:",
    test_df["timestamp"].min()
)

print(
    "No time leakage:",
    train_df["timestamp"].max()
    <= test_df["timestamp"].min()
)

Cleaned ratings: (100836, 4)
Training records: (80668, 4)
Testing records: (20168, 4)

Transformed datasets:
Training: (80668, 13)
Testing: (20168, 13)

Chronological split verification:
Training maximum timestamp: 1458635162
Testing minimum timestamp: 1458635171
No time leakage: True


In [ ]:
# feature store module
feature_store_code = '''
import os
import pandas as pd

def generate_user_features(train_df):
    """Generate user features from training data."""

    return train_df.groupby("userId").agg(
        user_avg_rating=("rating", "mean"),
        user_activity_count=("rating", "count")
    ).reset_index()


def generate_item_features(train_df):
    """Generate movie features from training data."""

    return train_df.groupby("movieId").agg(
        item_avg_rating=("rating", "mean"),
        item_popularity=("rating", "count"),
        unique_user_count=("userId", "nunique")
    ).reset_index()


def generate_interaction_features(train_df):
    """Generate interaction-level features."""

    columns = [
        "userId", "movieId", "rating",
        "timestamp", "year", "month", "day"
    ]

    return train_df[columns].copy()


def save_features(features, project_dir):
    """Save feature datasets in CSV and Parquet formats."""

    feature_dir = os.path.join(project_dir, "data", "features")
    os.makedirs(feature_dir, exist_ok=True)

    for name, dataframe in features.items():
        dataframe.to_csv(
            os.path.join(feature_dir, f"{name}.csv"),
            index=False
        )

        dataframe.to_parquet(
            os.path.join(feature_dir, f"{name}.parquet"),
            index=False
        )

    return True
'''

module_path = os.path.join(
    SRC_DIR, "features", "feature_store.py"
)

with open(module_path, "w") as file:
    file.write(feature_store_code)

print("Feature Store module created successfully!")

Feature Store module created successfully!


In [ ]:
# test feature geneartion
from src.features.feature_store import (
    generate_user_features,
    generate_item_features,
    generate_interaction_features
)

user_features = generate_user_features(train_transformed)
item_features = generate_item_features(train_transformed)
interaction_features = generate_interaction_features(train_transformed)

print("User features:", user_features.shape)
print("Item features:", item_features.shape)
print("Interaction features:", interaction_features.shape)

print("\nUser feature preview:")
display(user_features.head())

print("\nItem feature preview:")
display(item_features.head())

User features: (522, 3)
Item features: (7867, 4)
Interaction features: (80668, 7)

User feature preview:


,userId,user_avg_rating,user_activity_count
0,1,4.366379,232
1,2,3.948276,29
2,3,2.435897,39
3,4,3.555556,216
4,5,3.636364,44



Item feature preview:


,movieId,item_avg_rating,item_popularity,unique_user_count
0,1,3.923913,184,184
1,2,3.400000,95,95
2,3,3.294118,51,51
3,4,2.357143,7,7
4,5,3.063830,47,47


In [ ]:
#save features
from src.features.feature_store import save_features

features = {
    "user_features": user_features,
    "item_features": item_features,
    "interaction_features": interaction_features
}

save_features(features, PROJECT_DIR)

print("Feature datasets saved successfully!")

Feature datasets saved successfully!


In [ ]:
#verify saved features
feature_dir = os.path.join(PROJECT_DIR, "data", "features")

for name in [
    "user_features",
    "item_features",
    "interaction_features"
]:
    csv_path = os.path.join(feature_dir, f"{name}.csv")
    parquet_path = os.path.join(feature_dir, f"{name}.parquet")

    print(f"\n{name}")
    print("CSV exists:", os.path.exists(csv_path))
    print("Parquet exists:", os.path.exists(parquet_path))


user_features
CSV exists: True
Parquet exists: True

item_features
CSV exists: True
Parquet exists: True

interaction_features
CSV exists: True
Parquet exists: True


In [ ]:
#popularity model module
popularity_code = '''
import pandas as pd

def train_popularity_model(train_df, min_ratings=5):
    """Train a popularity-based movie recommendation model."""

    popularity = train_df.groupby("movieId").agg(
        average_rating=("rating", "mean"),
        rating_count=("rating", "count")
    ).reset_index()

    popularity = popularity[
        popularity["rating_count"] >= min_ratings
    ]

    popularity = popularity.sort_values(
        ["average_rating", "rating_count"],
        ascending=[False, False]
    ).reset_index(drop=True)

    return popularity


def recommend_popular_movies(
    popularity_model, seen_movie_ids=None, top_k=10
):
    """Recommend popular movies, excluding previously seen movies."""

    if seen_movie_ids is None:
        seen_movie_ids = set()

    recommendations = popularity_model[
        ~popularity_model["movieId"].isin(seen_movie_ids)
    ]

    return recommendations.head(top_k)["movieId"].tolist()
'''

module_path = os.path.join(
    SRC_DIR, "training", "popularity_model.py"
)

with open(module_path, "w") as file:
    file.write(popularity_code)

print("Popularity Model module created successfully!")

Popularity Model module created successfully!


In [ ]:
#Test Popularity Model module
from src.training.popularity_model import (
    train_popularity_model,
    recommend_popular_movies
)

popularity_model = train_popularity_model(
    train_transformed,
    min_ratings=5
)

print("Eligible movies:", len(popularity_model))

recommendations = recommend_popular_movies(
    popularity_model,
    top_k=10
)

print("Top 10 recommended movie IDs:", recommendations)

Eligible movies: 3123
Top 10 recommended movie IDs: [6460, 2239, 4334, 1041, 3068, 3201, 1104, 3451, 80906, 3152]


In [ ]:
# SVD model module
svd_code = '''
from surprise import Dataset, Reader, SVD

def train_svd_model(
    train_df,
    n_factors=50,
    n_epochs=20,
    random_state=42
):
    """Train a personalized SVD recommendation model."""

    reader = Reader(rating_scale=(0.5, 5.0))

    dataset = Dataset.load_from_df(
        train_df[["userId", "movieId", "rating"]],
        reader
    )

    trainset = dataset.build_full_trainset()

    model = SVD(
        n_factors=n_factors,
        n_epochs=n_epochs,
        lr_all=0.005,
        reg_all=0.02,
        random_state=random_state
    )

    model.fit(trainset)

    return model


def recommend_svd_movies(
    model, user_id, candidate_movie_ids,
    seen_movie_ids=None, top_k=10
):
    """Generate personalized movie recommendations."""

    if seen_movie_ids is None:
        seen_movie_ids = set()

    predictions = []

    for movie_id in candidate_movie_ids:
        if movie_id not in seen_movie_ids:
            score = model.predict(
                user_id, movie_id
            ).est

            predictions.append((movie_id, score))

    predictions.sort(key=lambda x: x[1], reverse=True)

    return [
        movie_id
        for movie_id, score in predictions[:top_k]
    ]
'''

module_path = os.path.join(
    SRC_DIR, "training", "svd_model.py"
)

with open(module_path, "w") as file:
    file.write(svd_code)

print("SVD Model module created successfully!")

SVD Model module created successfully!


In [ ]:
%pip install -q scikit-surprise

In [ ]:
#Verify SVD module
import joblib
from src.training.svd_model import recommend_svd_movies

svd_model_path = os.path.join(
    PROJECT_DIR, "models", "svd_model.pkl"
)

svd_model = joblib.load(svd_model_path)

candidate_movies = popularity_model["movieId"].tolist()

sample_user = train_transformed["userId"].iloc[0]

seen_movies = set(
    train_transformed.loc[
        train_transformed["userId"] == sample_user,
        "movieId"
    ]
)

svd_recommendations = recommend_svd_movies(
    svd_model,
    sample_user,
    candidate_movies,
    seen_movie_ids=seen_movies,
    top_k=10
)

print("Sample user:", sample_user)
print("SVD recommendations:", svd_recommendations)
print("Number of recommendations:", len(svd_recommendations))

Sample user: 429
SVD recommendations: [1104, 91529, 1204, 912, 750, 318, 2959, 3275, 930, 1089]
Number of recommendations: 10


In [ ]:
#evaluation metrics module
metrics_code = '''
import math

def precision_at_k(recommended, relevant, k=10):
    """Calculate Precision@K."""
    recommended = recommended[:k]
    relevant = set(relevant)

    hits = len(set(recommended) & relevant)
    return hits / k


def recall_at_k(recommended, relevant, k=10):
    """Calculate Recall@K."""
    relevant = set(relevant)

    if not relevant:
        return 0.0

    hits = len(set(recommended[:k]) & relevant)
    return hits / len(relevant)


def ndcg_at_k(recommended, relevant, k=10):
    """Calculate NDCG@K using binary relevance."""
    relevant = set(relevant)

    dcg = sum(
        1 / math.log2(rank + 2)
        for rank, movie_id in enumerate(recommended[:k])
        if movie_id in relevant
    )

    ideal_hits = min(len(relevant), k)

    idcg = sum(
        1 / math.log2(rank + 2)
        for rank in range(ideal_hits)
    )

    return dcg / idcg if idcg > 0 else 0.0


def catalog_coverage(all_recommendations, catalog_ids):
    """Calculate recommendation catalog coverage."""
    catalog_ids = set(catalog_ids)

    if not catalog_ids:
        return 0.0

    recommended_items = set()

    for recommendations in all_recommendations.values():
        recommended_items.update(recommendations)

    return len(recommended_items & catalog_ids) / len(catalog_ids)
'''

module_path = os.path.join(
    SRC_DIR, "evaluation", "metrics.py"
)

with open(module_path, "w") as file:
    file.write(metrics_code)

print("Evaluation metrics module created successfully!")

Evaluation metrics module created successfully!


In [ ]:
#Test evaluation metrics
from src.evaluation.metrics import (
    precision_at_k,
    recall_at_k,
    ndcg_at_k,
    catalog_coverage
)

sample_recommendations = [10, 20, 30, 40, 50]
sample_relevant = {20, 40, 60}

print("Precision@5:",
      precision_at_k(sample_recommendations, sample_relevant, k=5))

print("Recall@5:",
      recall_at_k(sample_recommendations, sample_relevant, k=5))

print("NDCG@5:",
      round(ndcg_at_k(sample_recommendations, sample_relevant, k=5), 4))

print("Coverage:",
      catalog_coverage(
          {1: [10, 20], 2: [20, 30]},
          {10, 20, 30, 40, 50}
      ))

Precision@5: 0.4
Recall@5: 0.6666666666666666
NDCG@5: 0.4982
Coverage: 0.6


In [ ]:
evaluation_code = '''
import pandas as pd

from .metrics import (
    precision_at_k,
    recall_at_k,
    ndcg_at_k,
    catalog_coverage
)


def evaluate_recommendations(
    recommendations_by_user,
    relevant_by_user,
    catalog_ids,
    k=10
):
    """Evaluate recommendations for users with relevant test items."""

    users = [
        user_id
        for user_id in recommendations_by_user
        if relevant_by_user.get(user_id)
    ]

    if not users:
        raise ValueError(
            "No eligible users with relevant test items."
        )

    precision_scores = []
    recall_scores = []
    ndcg_scores = []

    for user_id in users:
        recommended = recommendations_by_user[user_id]
        relevant = relevant_by_user[user_id]

        precision_scores.append(
            precision_at_k(recommended, relevant, k)
        )
        recall_scores.append(
            recall_at_k(recommended, relevant, k)
        )
        ndcg_scores.append(
            ndcg_at_k(recommended, relevant, k)
        )

    return {
        "Precision@10": sum(precision_scores) / len(users),
        "Recall@10": sum(recall_scores) / len(users),
        "NDCG@10": sum(ndcg_scores) / len(users),
        "Coverage": catalog_coverage(
            {u: recommendations_by_user[u] for u in users},
            catalog_ids
        ),
        "Evaluation_Users": len(users)
    }
'''

module_path = os.path.join(
    SRC_DIR, "evaluation", "evaluate.py"
)

with open(module_path, "w") as file:
    file.write(evaluation_code)

print("Model evaluation module created successfully!")

Model evaluation module created successfully!


In [ ]:
from src.evaluation.evaluate import evaluate_recommendations

sample_user_recommendations = {
    1: [10, 20, 30, 40, 50],
    2: [20, 30, 40, 50, 60]
}

sample_user_relevant = {
    1: {20, 40},
    2: {30, 60}
}

sample_catalog = {10, 20, 30, 40, 50, 60, 70}

results = evaluate_recommendations(
    sample_user_recommendations,
    sample_user_relevant,
    sample_catalog,
    k=5
)

print("Reusable evaluation module test:")
for metric, value in results.items():
    print(f"{metric}: {value:.4f}")

Reusable evaluation module test:
Precision@10: 0.4000
Recall@10: 1.0000
NDCG@10: 0.6375
Coverage: 0.8571
Evaluation_Users: 2.0000


In [ ]:
module_path = os.path.join(
    SRC_DIR, "evaluation", "evaluate.py"
)

with open(module_path, "r") as file:
    evaluation_code = file.read()

evaluation_code = evaluation_code.replace(
    '"Precision@10": sum(precision_scores) / len(users)',
    'f"Precision@{k}": sum(precision_scores) / len(users)'
).replace(
    '"Recall@10": sum(recall_scores) / len(users)',
    'f"Recall@{k}": sum(recall_scores) / len(users)'
).replace(
    '"NDCG@10": sum(ndcg_scores) / len(users)',
    'f"NDCG@{k}": sum(ndcg_scores) / len(users)'
)

with open(module_path, "w") as file:
    file.write(evaluation_code)

print("Evaluation metric labels updated successfully!")

Evaluation metric labels updated successfully!


In [ ]:
import importlib
import src.evaluation.evaluate as evaluation_module

importlib.reload(evaluation_module)

results = evaluation_module.evaluate_recommendations(
    sample_user_recommendations,
    sample_user_relevant,
    sample_catalog,
    k=5
)

for metric, value in results.items():
    print(f"{metric}: {value:.4f}")

Precision@5: 0.4000
Recall@5: 1.0000
NDCG@5: 0.6375
Coverage: 0.8571
Evaluation_Users: 2.0000


In [ ]:
import os
import yaml

CONFIG_PATH = os.path.join(
    PROJECT_DIR, "configs", "config.yaml"
)

print("Configuration file exists:", os.path.exists(CONFIG_PATH))

with open(CONFIG_PATH, "r") as file:
    config = yaml.safe_load(file)

print("\nConfiguration contents:")
print(yaml.safe_dump(config, sort_keys=False))

Configuration file exists: True

Configuration contents:
project:
  name: Movie Recommendation Pipeline
  version: '1.0'
  random_seed: 42
data:
  raw_dir: data/raw
  processed_dir: data/processed
  features_dir: data/features
  errors_dir: data/errors
  ratings_file: ratings.csv
  movies_file: movies.csv
  users_file: users.csv
  tags_file: tags.csv
validation:
  rating_min: 0.5
  rating_max: 5.0
  allow_missing_values: false
  check_duplicates: true
preparation:
  test_size: 0.2
  split_method: time_based
features:
  user_features:
  - user_avg_rating
  - user_activity_count
  item_features:
  - item_avg_rating
  - item_popularity
models:
  popularity:
    min_ratings: 5
  svd:
    n_factors: 50
    n_epochs: 20
    lr_all: 0.005
    reg_all: 0.02
    random_state: 42
evaluation:
  k: 10
  metrics:
  - Precision@10
  - Recall@10
  - NDCG@10
  - Coverage
  - Diversity
  - RMSE
  - MAE
storage:
  formats:
  - csv
  - parquet
mlflow:
  experiment_name: Movie_Recommendation_Experiments



In [ ]:
pipeline_code = '''
import os
import sys
import yaml

PROJECT_DIR = os.path.dirname(
    os.path.abspath(__file__)
)

sys.path.insert(0, PROJECT_DIR)

def verify_pipeline():
    print("Starting pipeline verification...")

    config_path = os.path.join(
        PROJECT_DIR, "configs", "config.yaml"
    )

    with open(config_path, "r") as file:
        config = yaml.safe_load(file)

    print("Configuration loaded successfully!")

    required_modules = [
        "src.ingestion.data_loader",
        "src.preparation.clean_data",
        "src.preparation.split_data",
        "src.transformation.transform_data",
        "src.features.feature_store",
        "src.training.popularity_model",
        "src.training.svd_model",
        "src.evaluation.metrics",
        "src.evaluation.evaluate"
    ]

    for module_name in required_modules:
        __import__(module_name)
        print("Verified:", module_name)

    required_files = [
        "data/raw/ratings.csv",
        "data/raw/movies.csv",
        "data/raw/users.csv",
        "models/popularity_model.pkl",
        "models/svd_model.pkl",
        "reports/model_evaluation_results.csv"
    ]

    for relative_path in required_files:
        file_path = os.path.join(
            PROJECT_DIR, relative_path
        )

        if not os.path.isfile(file_path):
            raise FileNotFoundError(
                f"Missing required file: {relative_path}"
            )

        print("Found:", relative_path)

    print("\\nPipeline verification completed successfully!")


if __name__ == "__main__":
    verify_pipeline()
'''

pipeline_path = os.path.join(
    PROJECT_DIR, "run_pipeline.py"
)

# Avoid silently overwriting an existing script
if os.path.exists(pipeline_path):
    print("run_pipeline.py already exists. No changes made.")
else:
    with open(pipeline_path, "w") as file:
        file.write(pipeline_code)

    print("Pipeline verification script created successfully!")

Pipeline verification script created successfully!


In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, pipeline_path],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("Errors or warnings:")
    print(result.stderr)

print("Exit code:", result.returncode)

Starting pipeline verification...
Configuration loaded successfully!
Verified: src.ingestion.data_loader
Verified: src.preparation.clean_data
Verified: src.preparation.split_data
Verified: src.transformation.transform_data
Verified: src.features.feature_store
Verified: src.training.popularity_model
Verified: src.training.svd_model
Verified: src.evaluation.metrics
Verified: src.evaluation.evaluate
Found: data/raw/ratings.csv
Found: data/raw/movies.csv
Found: data/raw/users.csv
Found: models/popularity_model.pkl
Found: models/svd_model.pkl
Found: reports/model_evaluation_results.csv

Pipeline verification completed successfully!

Exit code: 0


In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/recommendation-pipeline"

validation_path = os.path.join(
    PROJECT_DIR,
    "src",
    "validation",
    "validate_data.py"
)

print("Validation module exists:", os.path.isfile(validation_path))

if os.path.isfile(validation_path):
    print("\nCurrent validation module code:\n")

    with open(validation_path, "r") as file:
        print(file.read())

Validation module exists: True

Current validation module code:


import pandas as pd

from schema import (
    ratings_schema,
    users_schema,
    movies_schema,
    tags_schema,
    links_schema
)


def validate_dataset(dataframe, schema, dataset_name):
    try:
        validated_data = schema.validate(dataframe)

        print(f"{dataset_name}: VALIDATION PASSED")

        return validated_data

    except Exception as error:
        print(f"{dataset_name}: VALIDATION FAILED")
        print(error)

        return None



In [ ]:
import os
import yaml

config_path = os.path.join(
    PROJECT_DIR,
    "configs",
    "config.yaml"
)

print("Configuration file exists:", os.path.isfile(config_path))

with open(config_path, "r") as file:
    config = yaml.safe_load(file)

print("\nConfiguration sections:")

for key in config:
    print("-", key)

print("\nComplete configuration:\n")
print(yaml.safe_dump(config, sort_keys=False))

Configuration file exists: True

Configuration sections:
- project
- data
- validation
- preparation
- features
- models
- evaluation
- storage
- mlflow

Complete configuration:

project:
  name: Movie Recommendation Pipeline
  version: '1.0'
  random_seed: 42
data:
  raw_dir: data/raw
  processed_dir: data/processed
  features_dir: data/features
  errors_dir: data/errors
  ratings_file: ratings.csv
  movies_file: movies.csv
  users_file: users.csv
  tags_file: tags.csv
validation:
  rating_min: 0.5
  rating_max: 5.0
  allow_missing_values: false
  check_duplicates: true
preparation:
  test_size: 0.2
  split_method: time_based
features:
  user_features:
  - user_avg_rating
  - user_activity_count
  item_features:
  - item_avg_rating
  - item_popularity
models:
  popularity:
    min_ratings: 5
  svd:
    n_factors: 50
    n_epochs: 20
    lr_all: 0.005
    reg_all: 0.02
    random_state: 42
evaluation:
  k: 10
  metrics:
  - Precision@10
  - Recall@10
  - NDCG@10
  - Coverage
  - Divers

In [ ]:
import os

required_outputs = [
    "data/processed/train_data.parquet",
    "data/processed/test_data.parquet",
    "data/features/user_features.parquet",
    "data/features/item_features.parquet",
    "data/features/interaction_features.parquet",
    "models/popularity_model.pkl",
    "models/svd_model.pkl",
    "reports/model_evaluation_results.csv",
    "reports/feature_catalog.csv",
    "data/errors/invalid_ratings.csv",
    "mlflow.db"
]

missing_files = []

print("Checking pipeline output files:\n")

for relative_path in required_outputs:

    full_path = os.path.join(
        PROJECT_DIR,
        relative_path
    )

    exists = os.path.isfile(full_path)

    status = "FOUND" if exists else "MISSING"

    print(f"{relative_path}: {status}")

    if not exists:
        missing_files.append(relative_path)

print("\nVerification Summary")
print("--------------------")
print("Total files checked:", len(required_outputs))
print("Files found:", len(required_outputs) - len(missing_files))
print("Missing files:", len(missing_files))

if len(missing_files) == 0:
    print("\nAll pipeline output files verified successfully!")
else:
    print("\nMissing files:")
    for filename in missing_files:
        print("-", filename)

Checking pipeline output files:

data/processed/train_data.parquet: FOUND
data/processed/test_data.parquet: FOUND
data/features/user_features.parquet: FOUND
data/features/item_features.parquet: FOUND
data/features/interaction_features.parquet: FOUND
models/popularity_model.pkl: FOUND
models/svd_model.pkl: FOUND
reports/model_evaluation_results.csv: FOUND
reports/feature_catalog.csv: FOUND
data/errors/invalid_ratings.csv: FOUND
mlflow.db: FOUND

Verification Summary
--------------------
Total files checked: 11
Files found: 11
Missing files: 0

All pipeline output files verified successfully!


In [ ]:
# Creating pipeline script
pipeline_execution_code = '''
import os
import sys
import yaml
import logging

PROJECT_DIR = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, PROJECT_DIR)

from src.ingestion.data_loader import load_raw_data
from src.preparation.clean_data import clean_dataset
from src.preparation.split_data import time_based_split
from src.transformation.transform_data import transform_ratings
from src.features.feature_store import (
    generate_user_features,
    generate_item_features,
    generate_interaction_features
)

from src.validation.schema import (
    ratings_schema,
    movies_schema,
    users_schema,
    tags_schema,
    links_schema
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)


def run_pipeline(dry_run=True):

    config_path = os.path.join(
        PROJECT_DIR, "configs", "config.yaml"
    )

    with open(config_path, "r") as file:
        config = yaml.safe_load(file)

    logger.info("Pipeline started")
    logger.info("Dry run: %s", dry_run)

    # Stage 1: Ingestion
    datasets = load_raw_data(PROJECT_DIR)

    logger.info("Data ingestion completed")

    # Stage 2: Validation
    schemas = {
        "ratings": ratings_schema,
        "movies": movies_schema,
        "users": users_schema,
        "tags": tags_schema,
        "links": links_schema
    }

    for name, schema in schemas.items():
        schema.validate(datasets[name], lazy=True)
        logger.info("Validation passed: %s", name)

    # Stage 3: Cleaning
    ratings = clean_dataset(datasets["ratings"])
    movies = clean_dataset(datasets["movies"])
    users = clean_dataset(datasets["users"])

    logger.info("Data cleaning completed")

    # Stage 4: Time-based split
    test_size = config["preparation"]["test_size"]

    train_df, test_df = time_based_split(
        ratings,
        test_size=test_size
    )

    if train_df["timestamp"].max() > test_df["timestamp"].min():
        raise ValueError("Time leakage detected")

    logger.info(
        "Split completed: train=%s, test=%s",
        len(train_df),
        len(test_df)
    )

    # Stage 5: Transformation
    train_transformed = transform_ratings(
        train_df, movies, users
    )

    test_transformed = transform_ratings(
        test_df, movies, users
    )

    logger.info("Data transformation completed")

    # Stage 6: Feature generation
    user_features = generate_user_features(train_transformed)
    item_features = generate_item_features(train_transformed)
    interaction_features = generate_interaction_features(
        train_transformed
    )

    logger.info(
        "Feature generation completed: users=%s, items=%s, interactions=%s",
        len(user_features),
        len(item_features),
        len(interaction_features)
    )

    if dry_run:
        logger.info(
            "DRY RUN SUCCESSFUL - No files were overwritten"
        )

    return {
        "train_rows": len(train_transformed),
        "test_rows": len(test_transformed),
        "user_features": len(user_features),
        "item_features": len(item_features),
        "interaction_features": len(interaction_features)
    }


if __name__ == "__main__":
    result = run_pipeline(dry_run=True)
    print("\\nPIPELINE SUMMARY")
    for key, value in result.items():
        print(f"{key}: {value}")
'''

pipeline_execution_path = os.path.join(
    PROJECT_DIR, "pipeline_execution.py"
)

if os.path.exists(pipeline_execution_path):
    print("pipeline_execution.py already exists; no changes made.")
else:
    with open(pipeline_execution_path, "w") as file:
        file.write(pipeline_execution_code)

    print("Pipeline execution script created successfully!")

Pipeline execution script created successfully!


In [ ]:
%pip install -q "pandera[pandas]"

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, pipeline_execution_path],
    capture_output=True,
    text=True
)

print("PIPELINE OUTPUT:")
print(result.stdout)

print("\nPIPELINE LOGS:")
print(result.stderr)

print("\nExit code:", result.returncode)

PIPELINE OUTPUT:

PIPELINE SUMMARY
train_rows: 80668
test_rows: 20168
user_features: 522
item_features: 7867
interaction_features: 80668


PIPELINE LOGS:
2026-11-14 04:16:24,683 - INFO - Pipeline started
2026-11-14 04:16:24,683 - INFO - Dry run: True
2026-11-14 04:16:24,777 - INFO - Data ingestion completed
2026-11-14 04:16:25,237 - INFO - Validation passed: ratings
2026-11-14 04:16:25,253 - INFO - Validation passed: movies
2026-11-14 04:16:25,258 - INFO - Validation passed: users
2026-11-14 04:16:25,263 - INFO - Validation passed: tags
2026-11-14 04:16:25,265 - INFO - Validation passed: links
2026-11-14 04:16:25,297 - INFO - Data cleaning completed
2026-11-14 04:16:25,312 - INFO - Split completed: train=80668, test=20168
2026-11-14 04:16:25,383 - INFO - Data transformation completed
2026-11-14 04:16:25,406 - INFO - Feature generation completed: users=522, items=7867, interactions=80668
2026-11-14 04:16:25,406 - INFO - DRY RUN SUCCESSFUL - No files were overwritten


Exit code: 0


In [ ]:
#verify both trained model
import os
import joblib
import pandas as pd

models_dir = os.path.join(PROJECT_DIR, "models")

popularity_model = joblib.load(
    os.path.join(models_dir, "popularity_model.pkl")
)

svd_model = joblib.load(
    os.path.join(models_dir, "svd_model.pkl")
)

print("Popularity model loaded successfully!")
print("Eligible movies:", len(popularity_model))

print("\nSVD model loaded successfully!")
print("SVD model type:", type(svd_model).__name__)

Popularity model loaded successfully!
Eligible movies: 3123

SVD model loaded successfully!
SVD model type: SVD


In [ ]:
# verifying evaluation report
evaluation_path = os.path.join(
    PROJECT_DIR, "reports", "model_evaluation_results.csv"
)

evaluation_results = pd.read_csv(evaluation_path)

print("Existing model evaluation results:")
display(evaluation_results)

required_models = {"Popularity", "SVD"}

assert required_models.issubset(
    set(evaluation_results["Model"])
), "One or more models are missing from the evaluation report."

print("\nBoth model evaluation results verified successfully!")

Existing model evaluation results:


,Model,Precision@10,Recall@10,NDCG@10,Coverage,RMSE,MAE,Diversity
0,Popularity,0.001754,0.000111,0.001269,0.004163,NaN,NaN,0.368843
1,SVD,0.165789,0.042035,0.198545,0.030419,1.014546,0.782679,0.700039



Both model evaluation results verified successfully!


In [ ]:
#verify SVD prediction
test_path = os.path.join(
    PROJECT_DIR, "data", "processed", "test_data.parquet"
)

test_data = pd.read_parquet(test_path)

sample = test_data.iloc[0]

prediction = svd_model.predict(
    int(sample["userId"]),
    int(sample["movieId"])
)

print("User ID:", sample["userId"])
print("Movie ID:", sample["movieId"])
print("Actual rating:", sample["rating"])
print("Predicted rating:", round(prediction.est, 4))

assert 0.5 <= prediction.est <= 5.0

print("\nSVD prediction test passed!")

User ID: 495
Movie ID: 72998
Actual rating: 5.0
Predicted rating: 4.2387

SVD prediction test passed!


In [ ]:
#model integration module
model_integration_code = '''
import os
import joblib
import pandas as pd
from surprise import accuracy


def evaluate_saved_models(project_dir):
    """Load saved models and verify their evaluation results."""

    models_dir = os.path.join(project_dir, "models")

    popularity_model = joblib.load(
        os.path.join(models_dir, "popularity_model.pkl")
    )

    svd_model = joblib.load(
        os.path.join(models_dir, "svd_model.pkl")
    )

    test_df = pd.read_parquet(
        os.path.join(
            project_dir, "data", "processed", "test_data.parquet"
        )
    )

    testset = list(
        test_df[["userId", "movieId", "rating"]]
        .itertuples(index=False, name=None)
    )

    predictions = svd_model.test(testset)

    rmse = accuracy.rmse(predictions, verbose=False)
    mae = accuracy.mae(predictions, verbose=False)

    evaluation_report = pd.read_csv(
        os.path.join(
            project_dir, "reports", "model_evaluation_results.csv"
        )
    )

    if not {"Popularity", "SVD"}.issubset(
        set(evaluation_report["Model"])
    ):
        raise ValueError("Missing model evaluation results")

    return {
        "popularity_movies": len(popularity_model),
        "svd_rmse": rmse,
        "svd_mae": mae,
        "evaluated_test_rows": len(test_df),
        "evaluation_report": evaluation_report
    }
'''

module_path = os.path.join(
    SRC_DIR, "evaluation", "model_integration.py"
)

with open(module_path, "w") as file:
    file.write(model_integration_code)

print("Model integration module created successfully!")

Model integration module created successfully!


In [ ]:
#testing model integration module

from src.evaluation.model_integration import evaluate_saved_models

integration_results = evaluate_saved_models(PROJECT_DIR)

print("Model integration test results:")
print("Popularity eligible movies:",
      integration_results["popularity_movies"])

print("SVD RMSE:",
      round(integration_results["svd_rmse"], 4))

print("SVD MAE:",
      round(integration_results["svd_mae"], 4))

print("Evaluated test rows:",
      integration_results["evaluated_test_rows"])

print("\nExisting model comparison:")
display(integration_results["evaluation_report"])

print("\nModel integration test passed!")

Model integration test results:
Popularity eligible movies: 3123
SVD RMSE: 1.0145
SVD MAE: 0.7827
Evaluated test rows: 20168

Existing model comparison:


,Model,Precision@10,Recall@10,NDCG@10,Coverage,RMSE,MAE,Diversity
0,Popularity,0.001754,0.000111,0.001269,0.004163,NaN,NaN,0.368843
1,SVD,0.165789,0.042035,0.198545,0.030419,1.014546,0.782679,0.700039



Model integration test passed!


In [ ]:
#consistency with saved results
saved_svd = integration_results["evaluation_report"].loc[
    integration_results["evaluation_report"]["Model"] == "SVD"
].iloc[0]

rmse_difference = abs(
    integration_results["svd_rmse"] - saved_svd["RMSE"]
)

mae_difference = abs(
    integration_results["svd_mae"] - saved_svd["MAE"]
)

print("RMSE difference:", rmse_difference)
print("MAE difference:", mae_difference)

assert rmse_difference < 0.0001, "RMSE mismatch"
assert mae_difference < 0.0001, "MAE mismatch"

print("\nSaved and recalculated evaluation metrics match!")

RMSE difference: 0.0
MAE difference: 0.0

Saved and recalculated evaluation metrics match!


In [ ]:
import os
import shutil

pipeline_execution_path = os.path.join(
    PROJECT_DIR, "pipeline_execution.py"
)

# Create a backup before changing the script
backup_path = pipeline_execution_path + ".backup"

shutil.copy2(pipeline_execution_path, backup_path)

with open(pipeline_execution_path, "r") as file:
    pipeline_code = file.read()

# Add model evaluation import
new_import = (
    "from src.evaluation.model_integration "
    "import evaluate_saved_models\n"
)

if new_import not in pipeline_code:
    pipeline_code = pipeline_code.replace(
        "logging.basicConfig(",
        new_import + "\nlogging.basicConfig(",
        1
    )

# Add model evaluation stage before the dry-run summary
evaluation_stage = '''
    # Stage 7: Saved model evaluation
    model_results = evaluate_saved_models(PROJECT_DIR)

    logger.info(
        "Model evaluation completed: RMSE=%.4f, MAE=%.4f",
        model_results["svd_rmse"],
        model_results["svd_mae"]
    )

'''

if "model_results = evaluate_saved_models(PROJECT_DIR)" not in pipeline_code:
    pipeline_code = pipeline_code.replace(
        "    if dry_run:\n",
        evaluation_stage + "    if dry_run:\n",
        1
    )

# Add model metrics to the returned summary
old_summary = '"interaction_features": len(interaction_features)'
new_summary = '''"interaction_features": len(interaction_features),
        "svd_rmse": round(model_results["svd_rmse"], 4),
        "svd_mae": round(model_results["svd_mae"], 4)'''

if '"svd_rmse": round(model_results' not in pipeline_code:
    pipeline_code = pipeline_code.replace(
        old_summary,
        new_summary,
        1
    )

with open(pipeline_execution_path, "w") as file:
    file.write(pipeline_code)

print("Pipeline updated successfully!")
print("Backup created:", backup_path)

Pipeline updated successfully!
Backup created: /content/drive/MyDrive/recommendation-pipeline/pipeline_execution.py.backup


In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, pipeline_execution_path],
    capture_output=True,
    text=True
)

print("PIPELINE OUTPUT:")
print(result.stdout)

print("\nPIPELINE LOGS:")
print(result.stderr)

print("\nExit code:", result.returncode)

PIPELINE OUTPUT:

PIPELINE SUMMARY
train_rows: 80668
test_rows: 20168
user_features: 522
item_features: 7867
interaction_features: 80668
svd_rmse: 1.0145
svd_mae: 0.7827


PIPELINE LOGS:
2026-11-14 04:25:54,443 - INFO - Pipeline started
2026-11-14 04:25:54,444 - INFO - Dry run: True
2026-11-14 04:25:54,538 - INFO - Data ingestion completed
2026-11-14 04:25:54,643 - INFO - Validation passed: ratings
2026-11-14 04:25:54,657 - INFO - Validation passed: movies
2026-11-14 04:25:54,663 - INFO - Validation passed: users
2026-11-14 04:25:54,667 - INFO - Validation passed: tags
2026-11-14 04:25:54,669 - INFO - Validation passed: links
2026-11-14 04:25:54,702 - INFO - Data cleaning completed
2026-11-14 04:25:54,718 - INFO - Split completed: train=80668, test=20168
2026-11-14 04:25:54,788 - INFO - Data transformation completed
2026-11-14 04:25:54,812 - INFO - Feature generation completed: users=522, items=7867, interactions=80668
2026-11-14 04:25:55,930 - INFO - Model evaluation completed: RMSE=1

In [ ]:
#verify MLflow
%pip install -q mlflow

import mlflow
import os

MLFLOW_DB = os.path.join(PROJECT_DIR, "mlflow.db")

mlflow.set_tracking_uri(
    "sqlite:///" + MLFLOW_DB
)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("MLflow database exists:", os.path.isfile(MLFLOW_DB))

MLflow tracking URI: sqlite:////content/drive/MyDrive/recommendation-pipeline/mlflow.db
MLflow database exists: True


In [ ]:
mlflow_tracking_code = '''
import os
import mlflow


def log_pipeline_execution(project_dir, pipeline_results):
    """Log completed pipeline execution metrics to MLflow."""

    database_path = os.path.join(project_dir, "mlflow.db")

    mlflow.set_tracking_uri(
        "sqlite:///" + database_path
    )

    experiment_name = "MovieLens_Pipeline_Execution"

    mlflow.set_experiment(experiment_name)

    with mlflow.start_run(run_name="Pipeline_Dry_Run") as run:

        mlflow.log_param("execution_mode", "dry_run")
        mlflow.log_param("dataset", "MovieLens Latest Small")

        for metric_name, metric_value in pipeline_results.items():
            if isinstance(metric_value, (int, float)):
                mlflow.log_metric(
                    metric_name,
                    float(metric_value)
                )

        return run.info.run_id
'''

tracking_module_path = os.path.join(
    SRC_DIR, "evaluation", "mlflow_tracking.py"
)

with open(tracking_module_path, "w") as file:
    file.write(mlflow_tracking_code)

print("MLflow tracking module created successfully!")

MLflow tracking module created successfully!


In [ ]:
from src.evaluation.mlflow_tracking import log_pipeline_execution

test_pipeline_results = {
    "train_rows": 80668,
    "test_rows": 20168,
    "user_features": 522,
    "item_features": 7867,
    "interaction_features": 80668,
    "svd_rmse": 1.0145,
    "svd_mae": 0.7827
}

run_id = log_pipeline_execution(
    PROJECT_DIR,
    test_pipeline_results
)

print("Pipeline execution logged successfully!")
print("MLflow Run ID:", run_id)

2026/11/14 04:31:48 INFO mlflow.tracking.fluent: Experiment with name 'MovieLens_Pipeline_Execution' does not exist. Creating a new experiment.


Pipeline execution logged successfully!
MLflow Run ID: f14d98858cf64124a669496cdfc7018e


In [ ]:
import mlflow

experiment = mlflow.get_experiment_by_name(
    "MovieLens_Pipeline_Execution"
)

if experiment is None:
    print("Experiment not found.")
else:
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id]
    )

    print("Experiment:", experiment.name)
    print("Total runs:", len(runs))

    display(
        runs[
            [
                "run_id",
                "status",
                "metrics.train_rows",
                "metrics.test_rows",
                "metrics.svd_rmse",
                "metrics.svd_mae"
            ]
        ].head()
    )

Experiment: MovieLens_Pipeline_Execution
Total runs: 1


,run_id,status,metrics.train_rows,metrics.test_rows,metrics.svd_rmse,metrics.svd_mae
0,f14d98858cf64124a669496cdfc7018e,FINISHED,80668.0,20168.0,1.0145,0.7827


In [ ]:
#pipeline script

import os
import shutil
import ast

pipeline_execution_path = os.path.join(
    PROJECT_DIR, "pipeline_execution.py"
)

# Read the existing pipeline
with open(pipeline_execution_path, "r") as file:
    pipeline_code = file.read()

# Create a backup
backup_path = pipeline_execution_path + ".before_mlflow"

shutil.copy2(pipeline_execution_path, backup_path)

# Add the MLflow import if missing
tracking_import = (
    "from src.evaluation.mlflow_tracking "
    "import log_pipeline_execution"
)

if tracking_import not in pipeline_code:
    pipeline_code = (
        tracking_import + "\n" + pipeline_code
    )

# Find the main execution block using Python's syntax parser
tree = ast.parse(pipeline_code)

main_blocks = [
    node for node in tree.body
    if isinstance(node, ast.If)
    and isinstance(node.test, ast.Compare)
    and isinstance(node.test.left, ast.Name)
    and node.test.left.id == "__name__"
    and len(node.test.ops) == 1
    and isinstance(node.test.ops[0], ast.Eq)
    and len(node.test.comparators) == 1
    and isinstance(node.test.comparators[0], ast.Constant)
    and node.test.comparators[0].value == "__main__"
]

if len(main_blocks) != 1:
    raise ValueError(
        "Could not identify exactly one main execution block. "
        "No changes made."
    )

# Replace only the main block
lines = pipeline_code.splitlines(keepends=True)
main_block = main_blocks[0]

new_main = '''if __name__ == "__main__":
    result = run_pipeline(dry_run=True)

    print("\\nPIPELINE SUMMARY")
    for key, value in result.items():
        print(f"{key}: {value}")

    run_id = log_pipeline_execution(
        PROJECT_DIR, result
    )

    print("\\nMLflow tracking completed!")
    print("MLflow Run ID:", run_id)
'''

# Use a raw string to preserve the backslash in print("\\n...")
new_main = new_main.replace(
    'print("\nPIPELINE',
    'print("\\\\nPIPELINE'
).replace(
    'print("\nMLflow',
    'print("\\\\nMLflow'
)

updated_lines = (
    lines[:main_block.lineno - 1]
    + [new_main]
    + lines[main_block.end_lineno:]
)

updated_code = "".join(updated_lines)

# Check syntax before saving
compile(updated_code, pipeline_execution_path, "exec")

with open(pipeline_execution_path, "w") as file:
    file.write(updated_code)

print("Pipeline updated successfully with MLflow tracking!")
print("Backup saved at:", backup_path)
print("Python syntax verification passed!")

Pipeline updated successfully with MLflow tracking!
Backup saved at: /content/drive/MyDrive/recommendation-pipeline/pipeline_execution.py.before_mlflow
Python syntax verification passed!


In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, pipeline_execution_path],
    capture_output=True,
    text=True
)

print("PIPELINE OUTPUT:")
print(result.stdout)

print("\nPIPELINE LOGS:")
print(result.stderr)

print("\nExit code:", result.returncode)

PIPELINE OUTPUT:

PIPELINE SUMMARY
train_rows: 80668
test_rows: 20168
user_features: 522
item_features: 7867
interaction_features: 80668
svd_rmse: 1.0145
svd_mae: 0.7827

MLflow tracking completed!
MLflow Run ID: 4256176493d14b9a96d3da24dd07f398


PIPELINE LOGS:
2026-11-14 04:36:44,696 - INFO - Pipeline started
2026-11-14 04:36:44,696 - INFO - Dry run: True
2026-11-14 04:36:44,806 - INFO - Data ingestion completed
2026-11-14 04:36:44,913 - INFO - Validation passed: ratings
2026-11-14 04:36:44,929 - INFO - Validation passed: movies
2026-11-14 04:36:44,935 - INFO - Validation passed: users
2026-11-14 04:36:44,940 - INFO - Validation passed: tags
2026-11-14 04:36:44,943 - INFO - Validation passed: links
2026-11-14 04:36:44,987 - INFO - Data cleaning completed
2026-11-14 04:36:45,007 - INFO - Split completed: train=80668, test=20168
2026-11-14 04:36:45,091 - INFO - Data transformation completed
2026-11-14 04:36:45,119 - INFO - Feature generation completed: users=522, items=7867, interactio

In [ ]:
import os
import mlflow

# Connect to the existing MLflow database
mlflow.set_tracking_uri(
    "sqlite:///" + os.path.join(PROJECT_DIR, "mlflow.db")
)

# Find the pipeline execution experiment
experiment = mlflow.get_experiment_by_name(
    "MovieLens_Pipeline_Execution"
)

assert experiment is not None, "MLflow experiment not found!"

# Retrieve experiment runs
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

print("Experiment:", experiment.name)
print("Total runs:", len(runs))

# Display the important metrics
display(
    runs[
        [
            "run_id",
            "status",
            "metrics.train_rows",
            "metrics.test_rows",
            "metrics.svd_rmse",
            "metrics.svd_mae"
        ]
    ].head()
)

# Confirm that automatic logging succeeded
assert len(runs) >= 2, "Expected at least 2 MLflow runs!"

assert (
    runs["status"] == "FINISHED"
).sum() >= 2, "Expected at least 2 successful runs!"

print("\nStep 5L.3 PASSED!")
print("Automatic MLflow tracking verified successfully!")

Experiment: MovieLens_Pipeline_Execution
Total runs: 2


,run_id,status,metrics.train_rows,metrics.test_rows,metrics.svd_rmse,metrics.svd_mae
0,4256176493d14b9a96d3da24dd07f398,FINISHED,80668.0,20168.0,1.0145,0.7827
1,f14d98858cf64124a669496cdfc7018e,FINISHED,80668.0,20168.0,1.0145,0.7827



Step 5L.3 PASSED!
Automatic MLflow tracking verified successfully!


In [ ]:
#creating separate o/p directory
import os

FULL_RUN_DIR = os.path.join(
    PROJECT_DIR, "pipeline_runs", "full_run_test"
)

for folder in ["models", "reports", "features", "processed"]:
    os.makedirs(
        os.path.join(FULL_RUN_DIR, folder),
        exist_ok=True
    )

print("Full pipeline test directory created:")
print(FULL_RUN_DIR)

print("\nOutput folders:")
for folder in ["models", "reports", "features", "processed"]:
    print(folder, ":", os.path.isdir(
        os.path.join(FULL_RUN_DIR, folder)
    ))

Full pipeline test directory created:
/content/drive/MyDrive/recommendation-pipeline/pipeline_runs/full_run_test

Output folders:
models : True
reports : True
features : True
processed : True


In [ ]:
#test training both models
import os
import pandas as pd
import joblib

from src.training.popularity_model import train_popularity_model
from src.training.svd_model import train_svd_model

# Load the previously prepared training dataset
train_df = pd.read_parquet(
    os.path.join(
        PROJECT_DIR, "data", "processed", "train_data.parquet"
    )
)

print("Training records:", len(train_df))

# Train popularity baseline
new_popularity_model = train_popularity_model(
    train_df,
    min_ratings=5
)

print("Popularity model trained!")
print("Eligible movies:", len(new_popularity_model))

# Train a fresh SVD model
new_svd_model = train_svd_model(
    train_df,
    n_factors=50,
    n_epochs=20,
    random_state=42
)

print("SVD model trained!")

# Save both models to the isolated test directory
joblib.dump(
    new_popularity_model,
    os.path.join(FULL_RUN_DIR, "models", "popularity_model.pkl")
)

joblib.dump(
    new_svd_model,
    os.path.join(FULL_RUN_DIR, "models", "svd_model.pkl")
)

print("\nBoth newly trained models saved successfully!")

Training records: 80668
Popularity model trained!
Eligible movies: 3123
SVD model trained!

Both newly trained models saved successfully!


In [ ]:
#evaluating trained SVD model
import pandas as pd
from surprise import accuracy

test_df = pd.read_parquet(
    os.path.join(
        PROJECT_DIR, "data", "processed", "test_data.parquet"
    )
)

testset = list(
    test_df[["userId", "movieId", "rating"]]
    .itertuples(index=False, name=None)
)

new_predictions = new_svd_model.test(testset)

new_rmse = accuracy.rmse(
    new_predictions, verbose=False
)

new_mae = accuracy.mae(
    new_predictions, verbose=False
)

print("New SVD model evaluation:")
print("RMSE:", round(new_rmse, 6))
print("MAE:", round(new_mae, 6))
print("Test records:", len(test_df))

assert abs(new_rmse - 1.014546) < 0.001
assert abs(new_mae - 0.782679) < 0.001

print("\nNewly trained SVD model verified successfully!")

New SVD model evaluation:
RMSE: 1.014546
MAE: 0.782679
Test records: 20168

Newly trained SVD model verified successfully!


In [ ]:
import os

training_pipeline_code = '''
import os
import joblib

from surprise import accuracy

from src.training.popularity_model import train_popularity_model
from src.training.svd_model import train_svd_model


def execute_model_training(
    train_df,
    test_df,
    output_dir,
    config
):
    """Train and evaluate recommendation models."""

    models_dir = os.path.join(output_dir, "models")
    os.makedirs(models_dir, exist_ok=True)

    # Read model parameters from configuration
    popularity_config = config["models"]["popularity"]
    svd_config = config["models"]["svd"]

    # Train popularity model
    popularity_model = train_popularity_model(
        train_df,
        min_ratings=popularity_config["min_ratings"]
    )

    # Train SVD model
    svd_model = train_svd_model(
        train_df,
        n_factors=svd_config["n_factors"],
        n_epochs=svd_config["n_epochs"],
        random_state=svd_config["random_state"]
    )

    # Evaluate SVD on the testing dataset
    testset = list(
        test_df[["userId", "movieId", "rating"]]
        .itertuples(index=False, name=None)
    )

    predictions = svd_model.test(testset)

    rmse = accuracy.rmse(predictions, verbose=False)
    mae = accuracy.mae(predictions, verbose=False)

    # Save models in the selected output directory
    joblib.dump(
        popularity_model,
        os.path.join(models_dir, "popularity_model.pkl")
    )

    joblib.dump(
        svd_model,
        os.path.join(models_dir, "svd_model.pkl")
    )

    return {
        "popularity_movies": len(popularity_model),
        "svd_rmse": rmse,
        "svd_mae": mae,
        "training_rows": len(train_df),
        "testing_rows": len(test_df)
    }
'''

training_pipeline_path = os.path.join(
    SRC_DIR, "training", "training_pipeline.py"
)

with open(training_pipeline_path, "w") as file:
    file.write(training_pipeline_code)

print("Reusable training pipeline module created!")
print("Location:", training_pipeline_path)

Reusable training pipeline module created!
Location: /content/drive/MyDrive/recommendation-pipeline/src/training/training_pipeline.py


In [ ]:
#test reusable training module
import os
import yaml
import pandas as pd

from src.training.training_pipeline import execute_model_training

# Load configuration
with open(
    os.path.join(PROJECT_DIR, "configs", "config.yaml")
) as file:
    config = yaml.safe_load(file)

# Load training and testing data
train_df = pd.read_parquet(
    os.path.join(
        PROJECT_DIR, "data", "processed", "train_data.parquet"
    )
)

test_df = pd.read_parquet(
    os.path.join(
        PROJECT_DIR, "data", "processed", "test_data.parquet"
    )
)

# Use a separate directory for this test
TEST_OUTPUT_DIR = os.path.join(
    PROJECT_DIR, "pipeline_runs", "module_training_test"
)

training_results = execute_model_training(
    train_df,
    test_df,
    TEST_OUTPUT_DIR,
    config
)

print("Reusable training module results:")

for key, value in training_results.items():
    print(f"{key}: {value}")

print("\nTraining module executed successfully!")

Reusable training module results:
popularity_movies: 3123
svd_rmse: 1.014546085967989
svd_mae: 0.7826787900771747
training_rows: 80668
testing_rows: 20168

Training module executed successfully!


In [ ]:
#verify saved models files
import os
import joblib

model_files = [
    "popularity_model.pkl",
    "svd_model.pkl"
]

for filename in model_files:
    filepath = os.path.join(
        TEST_OUTPUT_DIR, "models", filename
    )

    assert os.path.isfile(filepath), (
        f"Missing model file: {filename}"
    )

    loaded_model = joblib.load(filepath)

    print(
        filename,
        "— saved and loaded successfully!",
        "Type:",
        type(loaded_model).__name__
    )

print("\n completed successfully!")

popularity_model.pkl — saved and loaded successfully! Type: DataFrame
svd_model.pkl — saved and loaded successfully! Type: SVD

 completed successfully!


In [ ]:
import os
import shutil
import yaml

#  Inspect existing SVD module
svd_path = os.path.join(
    SRC_DIR, "training", "svd_model.py"
)

# Create backup only if one does not already exist
backup_path = svd_path + ".backup"

if not os.path.exists(backup_path):
    shutil.copy2(svd_path, backup_path)

with open(svd_path, "r") as file:
    svd_code = file.read()

print("===== CURRENT SVD MODULE =====")
print(svd_code)

#  Inspect SVD configuration
with open(
    os.path.join(PROJECT_DIR, "configs", "config.yaml")
) as file:
    config = yaml.safe_load(file)

print("\n===== SVD CONFIGURATION =====")

for key, value in config["models"]["svd"].items():
    print(f"{key}: {value}")

print("\nStep inspection completed!")

===== CURRENT SVD MODULE =====

from surprise import Dataset, Reader, SVD

def train_svd_model(
    train_df,
    n_factors=50,
    n_epochs=20,
    random_state=42
):
    """Train a personalized SVD recommendation model."""

    reader = Reader(rating_scale=(0.5, 5.0))

    dataset = Dataset.load_from_df(
        train_df[["userId", "movieId", "rating"]],
        reader
    )

    trainset = dataset.build_full_trainset()

    model = SVD(
        n_factors=n_factors,
        n_epochs=n_epochs,
        lr_all=0.005,
        reg_all=0.02,
        random_state=random_state
    )

    model.fit(trainset)

    return model


def recommend_svd_movies(
    model, user_id, candidate_movie_ids,
    seen_movie_ids=None, top_k=10
):
    """Generate personalized movie recommendations."""

    if seen_movie_ids is None:
        seen_movie_ids = set()

    predictions = []

    for movie_id in candidate_movie_ids:
        if movie_id not in seen_movie_ids:
            score = model.predict(
       

In [ ]:
from pathlib import Path
import shutil
import ast
import importlib

svd_path = Path(SRC_DIR) / "training" / "svd_model.py"
training_path = Path(SRC_DIR) / "training" / "training_pipeline.py"

# Read existing modules
svd_code = svd_path.read_text()
training_code = training_path.read_text()

# Update SVD function parameters
old_signature = """    n_epochs=20,
    random_state=42
):"""

new_signature = """    n_epochs=20,
    random_state=42,
    lr_all=0.005,
    reg_all=0.02
):"""

old_parameters = """        lr_all=0.005,
        reg_all=0.02,
        random_state=random_state"""

new_parameters = """        lr_all=lr_all,
        reg_all=reg_all,
        random_state=random_state"""

# Update training pipeline to pass config values
old_call = """        n_epochs=svd_config["n_epochs"],
        random_state=svd_config["random_state"]"""

new_call = """        n_epochs=svd_config["n_epochs"],
        random_state=svd_config["random_state"],
        lr_all=svd_config["lr_all"],
        reg_all=svd_config["reg_all"]"""

# Verify existing code matches expected structure
if old_signature in svd_code and old_parameters in svd_code:
    svd_code = svd_code.replace(old_signature, new_signature, 1)
    svd_code = svd_code.replace(old_parameters, new_parameters, 1)
elif not (
    new_signature in svd_code
    and new_parameters in svd_code
):
    raise ValueError("Unexpected SVD module structure")

if old_call in training_code:
    training_code = training_code.replace(old_call, new_call, 1)
elif new_call not in training_code:
    raise ValueError("Unexpected training pipeline structure")

# Check syntax before writing
ast.parse(svd_code)
ast.parse(training_code)

# Back up both files
for path in [svd_path, training_path]:
    backup = path.with_suffix(".py.before_config")
    if not backup.exists():
        shutil.copy2(path, backup)

# Save changes
svd_path.write_text(svd_code)
training_path.write_text(training_code)

# Reload modules in Colab
import src.training.svd_model as svd_module
import src.training.training_pipeline as training_module

importlib.reload(svd_module)
importlib.reload(training_module)

print("SVD model updated successfully!")
print("Training pipeline updated successfully!")
print("Both Python modules passed syntax validation!")
print("Configuration parameters are now connected!")

SVD model updated successfully!
Training pipeline updated successfully!
Both Python modules passed syntax validation!
Configuration parameters are now connected!


In [ ]:
from pathlib import Path
import shutil
import ast

pipeline_path = Path(PROJECT_DIR) / "pipeline_execution.py"
code = pipeline_path.read_text()

# Add the reusable training import
training_import = (
    "from src.training.training_pipeline "
    "import execute_model_training"
)

if training_import not in code:
    code = training_import + "\n" + code

# Add full-run support before the existing dry-run summary
marker = "    if dry_run:\n"

full_run_stage = '''
    if not dry_run:
        output_dir = os.path.join(
            PROJECT_DIR, "pipeline_runs", "automated_full_run"
        )

        logger.info("Starting full model training")

        training_results = execute_model_training(
            train_df,
            test_df,
            output_dir,
            config
        )

        logger.info(
            "Training completed: RMSE=%.4f, MAE=%.4f",
            training_results["svd_rmse"],
            training_results["svd_mae"]
        )

        logger.info(
            "New models saved in: %s", output_dir
        )

'''

if "training_results = execute_model_training(" not in code:
    if marker not in code:
        raise ValueError("Pipeline insertion point not found")

    code = code.replace(marker, full_run_stage + marker, 1)

# Validate syntax before writing
ast.parse(code)

backup_path = pipeline_path.with_suffix(".py.before_full_run")

if not backup_path.exists():
    shutil.copy2(pipeline_path, backup_path)

pipeline_path.write_text(code)

print("Full-run training stage added successfully!")
print("Existing dry-run functionality preserved!")
print("Backup:", backup_path)

Full-run training stage added successfully!
Existing dry-run functionality preserved!
Backup: /content/drive/MyDrive/recommendation-pipeline/pipeline_execution.py.before_full_run


In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, str(pipeline_path)],
    capture_output=True,
    text=True
)

print("PIPELINE OUTPUT:")
print(result.stdout)

print("\nFINAL LOG MESSAGES:")
print("\n".join(result.stderr.splitlines()[-8:]))

print("\nExit code:", result.returncode)

assert result.returncode == 0, "Pipeline verification failed"

print("\nStep 5P verification successful!")

PIPELINE OUTPUT:

PIPELINE SUMMARY
train_rows: 80668
test_rows: 20168
user_features: 522
item_features: 7867
interaction_features: 80668
svd_rmse: 1.0145
svd_mae: 0.7827

MLflow tracking completed!
MLflow Run ID: 14de218edc3747c88e4257d8bbbcc5b4


FINAL LOG MESSAGES:
2026-11-14 04:56:06,206 - INFO - Validation passed: tags
2026-11-14 04:56:06,209 - INFO - Validation passed: links
2026-11-14 04:56:06,252 - INFO - Data cleaning completed
2026-11-14 04:56:06,273 - INFO - Split completed: train=80668, test=20168
2026-11-14 04:56:06,376 - INFO - Data transformation completed
2026-11-14 04:56:06,411 - INFO - Feature generation completed: users=522, items=7867, interactions=80668
2026-11-14 04:56:07,232 - INFO - Model evaluation completed: RMSE=1.0145, MAE=0.7827
2026-11-14 04:56:07,233 - INFO - DRY RUN SUCCESSFUL - No files were overwritten

Exit code: 0

Step 5P verification successful!


In [ ]:
from pathlib import Path

pipeline_path = Path(PROJECT_DIR) / "pipeline_execution.py"

print("===== CURRENT PIPELINE SCRIPT =====")
print(pipeline_path.read_text())

print("\n===== EXISTING EVALUATION FUNCTIONS =====")

evaluation_path = (
    Path(SRC_DIR) / "evaluation" / "evaluate.py"
)

metrics_path = (
    Path(SRC_DIR) / "evaluation" / "metrics.py"
)

print("\n--- evaluate.py ---")
print(evaluation_path.read_text())

print("\n--- metrics.py ---")
print(metrics_path.read_text())

===== CURRENT PIPELINE SCRIPT =====
from src.training.training_pipeline import execute_model_training
from src.evaluation.mlflow_tracking import log_pipeline_execution

import os
import sys
import yaml
import logging

PROJECT_DIR = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, PROJECT_DIR)

from src.ingestion.data_loader import load_raw_data
from src.preparation.clean_data import clean_dataset
from src.preparation.split_data import time_based_split
from src.transformation.transform_data import transform_ratings
from src.features.feature_store import (
    generate_user_features,
    generate_item_features,
    generate_interaction_features
)

from src.validation.schema import (
    ratings_schema,
    movies_schema,
    users_schema,
    tags_schema,
    links_schema
)

from src.evaluation.model_integration import evaluate_saved_models

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__nam

In [ ]:
from pathlib import Path
import shutil
import ast

pipeline_path = Path(PROJECT_DIR) / "pipeline_execution.py"
code = pipeline_path.read_text()

old_section = '''
    # Stage 7: Saved model evaluation
    model_results = evaluate_saved_models(PROJECT_DIR)

    logger.info(
        "Model evaluation completed: RMSE=%.4f, MAE=%.4f",
        model_results["svd_rmse"],
        model_results["svd_mae"]
    )


    if not dry_run:
        output_dir = os.path.join(
            PROJECT_DIR, "pipeline_runs", "automated_full_run"
        )

        logger.info("Starting full model training")

        training_results = execute_model_training(
            train_df,
            test_df,
            output_dir,
            config
        )

        logger.info(
            "Training completed: RMSE=%.4f, MAE=%.4f",
            training_results["svd_rmse"],
            training_results["svd_mae"]
        )

        logger.info(
            "New models saved in: %s", output_dir
        )
'''

new_section = '''
    # Stage 7: Model execution
    if dry_run:
        model_results = evaluate_saved_models(PROJECT_DIR)

        logger.info(
            "Saved model evaluation completed: RMSE=%.4f, MAE=%.4f",
            model_results["svd_rmse"],
            model_results["svd_mae"]
        )

    else:
        output_dir = os.path.join(
            PROJECT_DIR, "pipeline_runs", "automated_full_run"
        )

        logger.info("Starting full model training")

        model_results = execute_model_training(
            train_df,
            test_df,
            output_dir,
            config
        )

        logger.info(
            "Fresh model training completed: RMSE=%.4f, MAE=%.4f",
            model_results["svd_rmse"],
            model_results["svd_mae"]
        )

        logger.info("New models saved in: %s", output_dir)
'''

if old_section not in code:
    raise ValueError(
        "Expected pipeline section not found. No changes made."
    )

updated_code = code.replace(old_section, new_section, 1)

# Verify syntax before saving
ast.parse(updated_code)

# Back up existing script
backup = pipeline_path.with_suffix(".py.before_step5q")

if not backup.exists():
    shutil.copy2(pipeline_path, backup)

pipeline_path.write_text(updated_code)

print("Pipeline execution logic corrected!")
print("Dry run: evaluates existing models")
print("Full run: trains and evaluates new models")
print("Backup:", backup)

Pipeline execution logic corrected!
Dry run: evaluates existing models
Full run: trains and evaluates new models
Backup: /content/drive/MyDrive/recommendation-pipeline/pipeline_execution.py.before_step5q


In [ ]:
from pathlib import Path
import shutil
import ast

evaluation_path = Path(SRC_DIR) / "evaluation" / "evaluate.py"
existing_code = evaluation_path.read_text()

additional_code = '''

def evaluate_trained_models(
    train_df,
    test_df,
    popularity_model,
    svd_model,
    k=10,
    relevance_threshold=4.0
):
    """Evaluate both models on relevant test interactions."""

    import numpy as np

    from src.training.popularity_model import recommend_popular_movies
    from src.training.svd_model import recommend_svd_movies

    catalog_ids = popularity_model["movieId"].tolist()
    catalog_set = set(catalog_ids)

    # Movies already rated by each user during training
    seen_by_user = (
        train_df.groupby("userId")["movieId"]
        .agg(set)
        .to_dict()
    )

    # Relevant test movies
    relevant_by_user = (
        test_df[test_df["rating"] >= relevance_threshold]
        .groupby("userId")["movieId"]
        .agg(set)
        .to_dict()
    )

    # Restrict evaluation to users with relevant test items
    eligible_users = sorted(relevant_by_user)

    popularity_recommendations = {}
    svd_recommendations = {}

    for user_id in eligible_users:
        seen = seen_by_user.get(user_id, set())

        popularity_recommendations[user_id] = (
            recommend_popular_movies(
                popularity_model,
                seen_movie_ids=seen,
                top_k=k
            )
        )

        svd_recommendations[user_id] = (
            recommend_svd_movies(
                svd_model,
                user_id=user_id,
                candidate_movie_ids=catalog_ids,
                seen_movie_ids=seen,
                top_k=k
            )
        )

    results = []

    for model_name, recommendations in [
        ("Popularity", popularity_recommendations),
        ("SVD", svd_recommendations)
    ]:
        metrics = evaluate_recommendations(
            recommendations,
            relevant_by_user,
            catalog_set,
            k=k
        )

        # Diversity: mean pairwise genre dissimilarity
        # Requires item genres, added in the next integration step.
        metrics["Diversity"] = np.nan

        results.append({
            "Model": model_name,
            **metrics
        })

    return pd.DataFrame(results)
'''

if "def evaluate_trained_models(" not in existing_code:
    updated_code = existing_code + additional_code
else:
    updated_code = existing_code

ast.parse(updated_code)

backup = evaluation_path.with_suffix(".py.before_full_evaluation")

if not backup.exists():
    shutil.copy2(evaluation_path, backup)

evaluation_path.write_text(updated_code)

print("Full-model ranking evaluation function added!")
print("Existing evaluation functions preserved!")
print("Backup:", backup)

Full-model ranking evaluation function added!
Existing evaluation functions preserved!
Backup: /content/drive/MyDrive/recommendation-pipeline/src/evaluation/evaluate.py.before_full_evaluation


In [ ]:
import importlib
import src.evaluation.evaluate as evaluation_module

importlib.reload(evaluation_module)

assert hasattr(
    evaluation_module,
    "evaluate_trained_models"
)

print("Step 5R verification successful!")
print("Full-model evaluation function is available.")

Step 5R verification successful!
Full-model evaluation function is available.


In [ ]:
from pathlib import Path
import shutil
import ast

evaluation_path = Path(SRC_DIR) / "evaluation" / "evaluate.py"
code = evaluation_path.read_text()

old_code = '''        # Diversity: mean pairwise genre dissimilarity
        # Requires item genres, added in the next integration step.
        metrics["Diversity"] = np.nan'''

new_code = '''        # Diversity: average pairwise Jaccard genre distance
        movie_genres = movies_df.set_index("movieId")["genres"].to_dict()

        user_diversities = []

        for movie_list in recommendations.values():
            genre_sets = []

            for movie_id in movie_list[:k]:
                genres = movie_genres.get(movie_id, "")

                if pd.isna(genres) or genres == "(no genres listed)":
                    genre_sets.append(set())
                else:
                    genre_sets.append(set(genres.split("|")))

            distances = []

            for i in range(len(genre_sets)):
                for j in range(i + 1, len(genre_sets)):
                    union = genre_sets[i] | genre_sets[j]
                    intersection = genre_sets[i] & genre_sets[j]

                    distance = (
                        1 - len(intersection) / len(union)
                        if union else 0.0
                    )
                    distances.append(distance)

            if distances:
                user_diversities.append(
                    sum(distances) / len(distances)
                )

        metrics["Diversity"] = (
            sum(user_diversities) / len(user_diversities)
            if user_diversities else 0.0
        )'''

# Add movies_df as a function parameter
old_signature = '''    svd_model,
    k=10,'''

new_signature = '''    svd_model,
    movies_df,
    k=10,'''

if old_code not in code:
    raise ValueError("Expected Diversity placeholder not found")

if old_signature not in code:
    raise ValueError("Expected function signature not found")

updated_code = code.replace(old_code, new_code, 1)
updated_code = updated_code.replace(
    old_signature, new_signature, 1
)

# Validate before saving
ast.parse(updated_code)

# Back up existing evaluation module
backup = evaluation_path.with_suffix(".py.before_diversity")

if not backup.exists():
    shutil.copy2(evaluation_path, backup)

evaluation_path.write_text(updated_code)

print("Diversity calculation implemented successfully!")
print("Original genre-based Jaccard method reused!")
print("Evaluation module syntax verified!")

Diversity calculation implemented successfully!
Original genre-based Jaccard method reused!
Evaluation module syntax verified!


In [ ]:
#pipeline update
from pathlib import Path
import shutil
import ast

pipeline_path = Path(PROJECT_DIR) / "pipeline_execution.py"
code = pipeline_path.read_text()

# Add required imports
new_imports = """
import joblib
import pandas as pd
from src.evaluation.evaluate import evaluate_trained_models
"""

if "from src.evaluation.evaluate import evaluate_trained_models" not in code:
    code = new_imports + "\n" + code

# Find the full-run model training section
old_section = '''        logger.info("New models saved in: %s", output_dir)'''

new_section = '''        logger.info("New models saved in: %s", output_dir)

        # Load freshly trained models
        popularity_model = joblib.load(
            os.path.join(output_dir, "models", "popularity_model.pkl")
        )

        svd_model = joblib.load(
            os.path.join(output_dir, "models", "svd_model.pkl")
        )

        # Evaluate newly trained models
        evaluation_df = evaluate_trained_models(
            train_df,
            test_df,
            popularity_model,
            svd_model,
            movies,
            k=config["evaluation"]["k"]
        )

        # Include rating prediction metrics
        evaluation_df["RMSE"] = float("nan")
        evaluation_df["MAE"] = float("nan")

        svd_mask = evaluation_df["Model"] == "SVD"

        evaluation_df.loc[
            svd_mask, "RMSE"
        ] = model_results["svd_rmse"]

        evaluation_df.loc[
            svd_mask, "MAE"
        ] = model_results["svd_mae"]

        # Save fresh evaluation report
        reports_dir = os.path.join(output_dir, "reports")
        os.makedirs(reports_dir, exist_ok=True)

        report_path = os.path.join(
            reports_dir, "model_evaluation_results.csv"
        )

        evaluation_df.to_csv(report_path, index=False)

        logger.info("Fresh evaluation report saved: %s", report_path)

        # Include fresh ranking metrics in pipeline results
        for _, row in evaluation_df.iterrows():
            prefix = row["Model"].lower()

            for metric in [
                "Precision@10",
                "Recall@10",
                "NDCG@10",
                "Coverage",
                "Diversity"
            ]:
                model_results[
                    f"{prefix}_{metric.replace('@', '_at_')}"
                ] = float(row[metric])'''

if old_section not in code:
    raise ValueError(
        "Expected full-run section not found. No changes made."
    )

code = code.replace(old_section, new_section, 1)

# Ensure full-run metrics are included in the returned summary
old_return = '''    return {
        "train_rows": len(train_transformed),'''

new_return = '''    result = {
        "train_rows": len(train_transformed),'''

if old_return not in code:
    raise ValueError("Expected return block not found")

code = code.replace(old_return, new_return, 1)

old_end = '''        "svd_mae": round(model_results["svd_mae"], 4)
    }'''

new_end = '''        "svd_mae": round(model_results["svd_mae"], 4)
    }

    if not dry_run:
        for key, value in model_results.items():
            if key.startswith(("popularity_", "svd_")):
                if isinstance(value, (int, float)):
                    result[key] = value

    return result'''

if old_end not in code:
    raise ValueError("Expected summary ending not found")

code = code.replace(old_end, new_end, 1)

# Validate syntax before saving
ast.parse(code)

backup = pipeline_path.with_suffix(".py.before_step5t")

if not backup.exists():
    shutil.copy2(pipeline_path, backup)

pipeline_path.write_text(code)

print("Full evaluation connected to the pipeline!")
print("All seven metrics included!")
print("Fresh evaluation report will be saved separately!")
print("Existing dry-run functionality preserved!")

Full evaluation connected to the pipeline!
All seven metrics included!
Fresh evaluation report will be saved separately!
Existing dry-run functionality preserved!


In [ ]:
## Verify the full pipeline structure ##
from pathlib import Path
import ast
import importlib
import inspect

pipeline_path = Path(PROJECT_DIR) / "pipeline_execution.py"

# Verify Python syntax
code = pipeline_path.read_text()
ast.parse(code)

# Check required pipeline components
required_components = [
    "execute_model_training",
    "evaluate_trained_models",
    "log_pipeline_execution",
    "evaluation_df.to_csv",
    "run_pipeline(dry_run=True)"
]

for component in required_components:
    assert component in code, f"Missing: {component}"
    print("Verified:", component)

# Verify evaluation function parameters
import src.evaluation.evaluate as evaluation_module

importlib.reload(evaluation_module)

parameters = inspect.signature(
    evaluation_module.evaluate_trained_models
).parameters

assert "movies_df" in parameters
assert "svd_model" in parameters

print("\nPipeline structure verification PASSED!")

Verified: execute_model_training
Verified: evaluate_trained_models
Verified: log_pipeline_execution
Verified: evaluation_df.to_csv
Verified: run_pipeline(dry_run=True)

Pipeline structure verification PASSED!


In [ ]:
#enable full run mode
from pathlib import Path
import shutil
import ast

pipeline_path = Path(PROJECT_DIR) / "pipeline_execution.py"
code = pipeline_path.read_text()

old_line = "    result = run_pipeline(dry_run=True)"

new_line = """    dry_run = "--full-run" not in sys.argv
    result = run_pipeline(dry_run=dry_run)"""

if old_line not in code:
    raise ValueError(
        "Expected execution line not found. No changes made."
    )

updated_code = code.replace(old_line, new_line, 1)

ast.parse(updated_code)

backup_path = pipeline_path.with_suffix(".py.before_step5u")
shutil.copy2(pipeline_path, backup_path)

pipeline_path.write_text(updated_code)

print("Full-run command enabled successfully!")
print("Dry run remains the default.")
print("Backup:", backup_path)

Full-run command enabled successfully!
Dry run remains the default.
Backup: /content/drive/MyDrive/recommendation-pipeline/pipeline_execution.py.before_step5u


In [ ]:
import subprocess
import sys
import time

print("Starting full end-to-end pipeline execution...")

start_time = time.time()

result = subprocess.run(
    [
        sys.executable,
        str(pipeline_path),
        "--full-run"
    ],
    capture_output=True,
    text=True
)

elapsed = time.time() - start_time

print("\n===== PIPELINE OUTPUT =====")
print(result.stdout)

print("\n===== PIPELINE LOGS =====")
print(result.stderr[-6000:])

print("\nExit code:", result.returncode)
print("Execution time (minutes):", round(elapsed / 60, 2))

if result.returncode == 0:
    print("\nFULL PIPELINE EXECUTION SUCCESSFUL!")
else:
    print("\nPipeline failed. Please share the error logs.")

Starting full end-to-end pipeline execution...

===== PIPELINE OUTPUT =====

PIPELINE SUMMARY
train_rows: 80668
test_rows: 20168
user_features: 522
item_features: 7867
interaction_features: 80668
svd_rmse: 1.0142989732383623
svd_mae: 0.7825417052894287
popularity_movies: 3123
popularity_Precision_at_10: 0.001724137931034483
popularity_Recall_at_10: 9.730391620930183e-05
popularity_NDCG_at_10: 0.0012470042451126237
popularity_Coverage: 0.004162664105027218
popularity_Diversity: 0.36832694763729246
svd_Precision_at_10: 0.15862068965517243
svd_Recall_at_10: 0.03322691042097276
svd_NDCG_at_10: 0.1870414561099631
svd_Coverage: 0.03170028818443804
svd_Diversity: 0.7467871586693426

MLflow tracking completed!
MLflow Run ID: 4336483410d0473cbd41829052ab2ad3


===== PIPELINE LOGS =====
2026-11-14 05:18:30,401 - INFO - Pipeline started
2026-11-14 05:18:30,402 - INFO - Dry run: False
2026-11-14 05:18:30,540 - INFO - Data ingestion completed
2026-11-14 05:18:30,706 - INFO - Validation passed: rati

In [ ]:
from pathlib import Path
import ast
import shutil
import re

project = Path(PROJECT_DIR)
pipeline_path = project / "pipeline_execution.py"
tracking_path = project / "src/evaluation/mlflow_tracking.py"

pipeline = pipeline_path.read_text()
tracking = tracking_path.read_text()

# 1. Update MLflow tracking function
tracking = tracking.replace(
    "def log_pipeline_execution(project_dir, pipeline_results):",
    "def log_pipeline_execution(project_dir, pipeline_results, dry_run=True):",
    1
)

tracking = tracking.replace(
    'run_name="Pipeline_Dry_Run"',
    'run_name="Pipeline_Dry_Run" if dry_run else "Pipeline_Full_Run"',
    1
)

tracking = tracking.replace(
    'mlflow.log_param("execution_mode", "dry_run")',
    'mlflow.log_param("execution_mode", "dry_run" if dry_run else "full_run")',
    1
)

# 2. Handle multi-line MLflow calls
pattern = r'log_pipeline_execution\(\s*PROJECT_DIR\s*,\s*result\s*\)'

pipeline, count = re.subn(
    pattern,
    "log_pipeline_execution(PROJECT_DIR, result, dry_run=dry_run)",
    pipeline,
    count=1
)

if count != 1:
    raise ValueError(
        "MLflow call has a different structure. No files changed."
    )

# 3. Add output saving in full-run mode
anchor = '        logger.info("Starting full model training")'

save_code = '''        # Save processed datasets and generated features
        data_dir = os.path.join(output_dir, "data", "processed")
        features_dir = os.path.join(output_dir, "data", "features")

        os.makedirs(data_dir, exist_ok=True)
        os.makedirs(features_dir, exist_ok=True)

        train_df.to_parquet(
            os.path.join(data_dir, "train_data.parquet"),
            index=False
        )
        test_df.to_parquet(
            os.path.join(data_dir, "test_data.parquet"),
            index=False
        )

        for name, df in {
            "user_features": user_features,
            "item_features": item_features,
            "interaction_features": interaction_features
        }.items():
            df.to_parquet(
                os.path.join(features_dir, f"{name}.parquet"),
                index=False
            )

        logger.info("Fresh processed data and features saved")
'''

if anchor not in pipeline:
    raise ValueError(
        "Full-run training section not found. No files changed."
    )

pipeline = pipeline.replace(anchor, save_code + anchor, 1)

# 4. Verify both updated files before saving
ast.parse(pipeline)
ast.parse(tracking)

assert "dry_run=dry_run" in pipeline
assert '"full_run"' in tracking

# 5. Create backups
shutil.copy2(
    pipeline_path,
    pipeline_path.with_suffix(".py.before_step5v")
)

shutil.copy2(
    tracking_path,
    tracking_path.with_suffix(".py.before_step5v")
)

# 6. Save updates
pipeline_path.write_text(pipeline)
tracking_path.write_text(tracking)

print("Step 5V.1 completed successfully!")
print("MLflow full-run labeling corrected.")
print("Automatic data and feature saving enabled.")
print("Both modules passed syntax validation.")
print("Existing models and evaluation results unchanged.")

Step 5V.1 completed successfully!
MLflow full-run labeling corrected.
Automatic data and feature saving enabled.
Both modules passed syntax validation.
Existing models and evaluation results unchanged.


In [ ]:
from pathlib import Path
import shutil
import pandas as pd

project = Path(PROJECT_DIR)

output_dir = project / "pipeline_runs" / "automated_full_run"
data_dir = output_dir / "data" / "processed"
features_dir = output_dir / "data" / "features"

data_dir.mkdir(parents=True, exist_ok=True)
features_dir.mkdir(parents=True, exist_ok=True)

# Source files from previously completed notebook executions
files_to_copy = {
    project / "data/processed/train_data.parquet":
        data_dir / "train_data.parquet",

    project / "data/processed/test_data.parquet":
        data_dir / "test_data.parquet",

    project / "data/features/user_features.parquet":
        features_dir / "user_features.parquet",

    project / "data/features/item_features.parquet":
        features_dir / "item_features.parquet",

    project / "data/features/interaction_features.parquet":
        features_dir / "interaction_features.parquet"
}

# Check all source files before copying anything
missing = [
    str(source)
    for source in files_to_copy
    if not source.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing source files:\n" + "\n".join(missing)
    )

# Copy datasets and features only
for source, destination in files_to_copy.items():
    shutil.copy2(source, destination)
    print("Saved:", destination.relative_to(output_dir))

# Verify successful full-run artifacts
required_files = [
    output_dir / "models/popularity_model.pkl",
    output_dir / "models/svd_model.pkl",
    output_dir / "reports/model_evaluation_results.csv",
    *files_to_copy.values()
]

missing_outputs = [
    str(path)
    for path in required_files
    if not path.exists()
]

if missing_outputs:
    raise FileNotFoundError(
        "Missing output files:\n" + "\n".join(missing_outputs)
    )

print("\nAll required full-run artifacts verified!")
print("No model retraining performed.")
print("Existing models and evaluation report preserved.")
print("Step completed successfully!")

Saved: data/processed/train_data.parquet
Saved: data/processed/test_data.parquet
Saved: data/features/user_features.parquet
Saved: data/features/item_features.parquet
Saved: data/features/interaction_features.parquet

All required full-run artifacts verified!
No model retraining performed.
Existing models and evaluation report preserved.
Step completed successfully!
